# 01 — Dataset Exploration

Overview of the historical AQI dataset generated from Open-Meteo APIs.

**Dataset:** 107,064 hourly observations across 3 Pakistani cities (Aug 2022 – Aug 2026)  
**Sources:** Open-Meteo Weather Archive + CAMS Air Quality  
**Target:** US EPA PM AQI (0–500 scale)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/processed/raw_observations.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

print(f"Rows: {len(df):,}")
print(f"Cities: {df['location_id'].unique().tolist()}")
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Columns: {len(df.columns)}")
df.head(3)

## 2. AQI Distribution

In [ ]:
aqi = df['aqi'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(aqi, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('AQI Distribution (All Cities)')
axes[0].set_xlabel('AQI')
axes[0].set_ylabel('Count')
axes[0].axvline(aqi.median(), color='red', linestyle='--', label=f'Median={aqi.median():.0f}')
axes[0].legend()

# Category breakdown
from src.utils.aqi_categories import get_aqi_category
cats = aqi.apply(lambda x: get_aqi_category(int(x))[1])
cat_order = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
cat_counts = cats.value_counts().reindex(cat_order).fillna(0)
colors = ['#00E400', '#FFFF00', '#FF7E00', '#FF0000', '#8F3F97', '#7E0023']
axes[1].barh(cat_order, cat_counts.values, color=colors)
axes[1].set_title('AQI Category Distribution')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

print(f"\nStatistics: mean={aqi.mean():.1f}, median={aqi.median():.0f}, std={aqi.std():.1f}, min={aqi.min():.0f}, max={aqi.max():.0f}")

## 3. City Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cities = ['karachi', 'lahore', 'islamabad']
city_colors = ['#2196F3', '#FF5722', '#4CAF50']

for ax, city, color in zip(axes, cities, city_colors):
    city_aqi = df[df['location_id'] == city]['aqi'].dropna()
    ax.hist(city_aqi, bins=40, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(f'{city.title()} (n={len(city_aqi):,})')
    ax.set_xlabel('AQI')
    ax.set_ylabel('Count')
    ax.axvline(city_aqi.mean(), color='black', linestyle='--', alpha=0.7,
               label=f'Mean={city_aqi.mean():.0f}')
    ax.legend(fontsize=9)

plt.suptitle('AQI Distribution by City', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Per-city statistics table
stats = []
for city in cities:
    c = df[df['location_id'] == city]['aqi'].dropna()
    stats.append({
        'City': city.title(),
        'n': len(c),
        'Mean': f'{c.mean():.1f}',
        'Median': f'{c.median():.0f}',
        'Std': f'{c.std():.1f}',
        'Min': f'{c.min():.0f}',
        'Max': f'{c.max():.0f}',
        'P95': f'{c.quantile(0.95):.0f}',
    })
pd.DataFrame(stats)

## 4. Missing Values

In [ ]:
key_cols = ['temperature', 'humidity', 'pressure', 'wind_speed', 'pm25', 'pm10', 'co', 'no2', 'so2', 'o3', 'aqi']
missing = df[key_cols].isna().sum()
missing_pct = (missing / len(df) * 100).round(2)

miss_df = pd.DataFrame({'Missing': missing, 'Pct': missing_pct})
miss_df = miss_df[miss_df['Missing'] > 0]

if len(miss_df) > 0:
    miss_df.plot(kind='bar', y='Missing', legend=False, color='coral')
    plt.title('Missing Values by Column')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()
else:
    print('No missing values found.')

print(miss_df)

## 5. Pollutant Analysis

In [ ]:
pollutants = ['pm25', 'pm10', 'co', 'no2', 'so2', 'o3']
units = ['μg/m³', 'μg/m³', 'μg/m³', 'μg/m³', 'μg/m³', 'μg/m³']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col, unit in zip(axes.flat, pollutants, units):
    for city, color in zip(cities, city_colors):
        vals = df[df['location_id'] == city][col].dropna()
        ax.hist(vals, bins=40, alpha=0.5, label=city.title(), color=color, density=True)
    ax.set_title(f'{col.upper()} ({unit})')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Pollutant Distributions by City', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Temporal Patterns

In [ ]:
# Monthly AQI trend
df['month'] = df['timestamp'].dt.to_period('M')
monthly = df.groupby(['month', 'location_id'])['aqi'].mean().unstack()

fig, ax = plt.subplots(figsize=(14, 4))
for city, color in zip(cities, city_colors):
    if city in monthly.columns:
        monthly[city].plot(ax=ax, label=city.title(), color=color, linewidth=1.5)
ax.set_title('Monthly Average AQI Trend')
ax.set_ylabel('Mean AQI')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal pattern
df['hour'] = df['timestamp'].dt.hour
df['month_num'] = df['timestamp'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Hourly pattern
hourly = df.groupby('hour')['aqi'].mean()
axes[0].plot(hourly.index, hourly.values, 'o-', color='steelblue')
axes[0].set_title('Average AQI by Hour of Day')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('Mean AQI')

# Monthly pattern
monthly_avg = df.groupby('month_num')['aqi'].mean()
axes[1].bar(monthly_avg.index, monthly_avg.values, color='steelblue', edgecolor='white')
axes[1].set_title('Average AQI by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Mean AQI')

plt.tight_layout()
plt.show()

## 7. Weather vs AQI

In [ ]:
weather_cols = ['temperature', 'humidity', 'wind_speed', 'pressure']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, col in zip(axes, weather_cols):
    for city, color in zip(cities, city_colors):
        subset = df[df['location_id'] == city][[col, 'aqi']].dropna()
        ax.scatter(subset[col], subset['aqi'], alpha=0.05, s=5, color=color, label=city.title())
    ax.set_title(f'{col.title()} vs AQI')
    ax.set_xlabel(col.title())
    ax.set_ylabel('AQI')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()